Problem 3

In [1]:
!wget http://go.criteo.net/criteo-research-attribution-dataset.zip -O criteo.zip

--2025-11-18 16:12:02--  http://go.criteo.net/criteo-research-attribution-dataset.zip
Resolving go.criteo.net (go.criteo.net)... 74.119.117.38, 2620:100:a00b::27
Connecting to go.criteo.net (go.criteo.net)|74.119.117.38|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://criteostorage.blob.core.windows.net/criteo-research-datasets/criteo_attribution_dataset.zip [following]
--2025-11-18 16:12:03--  https://criteostorage.blob.core.windows.net/criteo-research-datasets/criteo_attribution_dataset.zip
Resolving criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)... 20.209.1.1
Connecting to criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)|20.209.1.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 653128946 (623M) [application/zip]
Saving to: ‘criteo.zip’

criteo.zip          100%[===================>] 622.87M  21.8MB/s    in 30s     

2025-11-18 16:12:33 (20.9 MB/s) - ‘criteo.zip’ saved 

In [2]:
!unzip criteo.zip -d criteo_data

Archive:  criteo.zip
  inflating: criteo_data/Experiments.ipynb  
  inflating: criteo_data/README.md   
  inflating: criteo_data/criteo_attribution_dataset.tsv.gz  


In [3]:
import duckdb

con = duckdb.connect()

df = con.execute("""
    SELECT *
    FROM read_csv_auto(
        '/content/criteo_data/criteo_attribution_dataset.tsv.gz',
        delim='\t'
    )
    LIMIT 50000
""").df()

df.head()

,timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,...,time_since_last_click,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9
0,0,20073966,22589171,0,-1,-1,0,0,-1,-1,...,-1,5824233,9312274,3490278,29196072,11409686,1973606,25162884,29196072,29196072
1,2,24607497,884761,0,-1,-1,0,0,-1,-1,...,423858,30763035,9312274,14584482,29196072,11409686,1973606,22644417,9312274,21091111
2,2,28474333,18975823,0,-1,-1,0,0,-1,-1,...,8879,138937,9312274,10769841,29196072,5824237,138937,1795451,29196072,15351056
3,3,7306395,29427842,1,1449193,3063962,0,1,0,7,...,-1,28928366,26597095,12435261,23549932,5824237,1973606,9180723,29841067,29196072
4,3,25357769,13365547,0,-1,-1,0,0,-1,-1,...,-1,138937,26597094,31616034,29196072,11409684,26597096,4480345,29196072,29196072


In [4]:
df.columns

Index(['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp',
       'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb',
       'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4',
       'cat5', 'cat6', 'cat7', 'cat8', 'cat9'],
      dtype='object')

In [5]:
import numpy as np
df['day'] = np.floor(df.timestamp / 86400.).astype(int)
df['hour'] = ((df.timestamp % 86400) // 3600).astype(int)

In [6]:
cols = ['campaign', 'click', 'click_pos', 'click_nb', 'cost',
        'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4',
        'cat5', 'cat6', 'cat7', 'cat8', 'cat9', 'hour', 'day']
df = df[cols]

In [7]:
cat_cols = ['campaign','cat1','cat2','cat3','cat4','cat5','cat6','cat7','cat8','cat9']

for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes

In [8]:
df.columns

Index(['campaign', 'click', 'click_pos', 'click_nb', 'cost',
       'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6',
       'cat7', 'cat8', 'cat9', 'hour', 'day'],
      dtype='object')

In [9]:
import lightgbm as lgb
model = lgb.Booster(model_file='pctr_baseline.txt')

X = df.drop(columns=['click', 'cost'])
pctr = model.predict(X)

In [10]:
X

,campaign,click_pos,click_nb,time_since_last_click,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,hour,day
0,418,-1,-1,-1,3,19,163,14,16,1,4269,8,25,0,0
1,17,-1,-1,423858,8,19,678,14,16,1,3863,3,19,0,0
2,373,-1,-1,8879,0,19,517,14,7,0,280,8,11,0,0
3,562,0,7,-1,7,32,579,11,7,1,1529,9,25,0,0
4,261,-1,-1,-1,0,31,1365,14,14,18,724,8,25,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,431,-1,-1,-1,8,19,1378,14,47,22,1573,8,25,6,0
49996,202,-1,-1,1357839,5,14,1220,7,46,1,5321,4,17,6,0
49997,121,-1,-1,1458438,8,19,8,14,46,1,4370,8,17,6,0
49998,519,-1,-1,-1,0,19,503,14,10,21,1251,8,25,6,0


# **Multi-Objective Optimization for Ad Allocation**

This section describes the formulation of a multi-objective optimization problem for selecting ad impressions under a budget constraint using predicted click-through rates (CTR), conversion proxies, and regularization.

---

# ## 1. What is *q*? (Conversion Proxy Signal)

In online advertising, **conversion rates (CVR)** are typically *much rarer* than clicks.
If you don't have true conversions, you approximate "conversion intent" using **proxy features**.

For this dataset, we could also get a pCvr instead by training a model to predict conversion rates.

### In this dataset, a natural conversion proxy is:

$$
q_i = \frac{1}{\text{time_since_last_click}_i + 1}
$$

Interpretation:

* If a user clicked recently → high q
* If a user has not clicked for a long time → low q
* If the feature is `-1` (meaning "no click history") → treat as a very low conversion likelihood

This is similar to **recency scoring** in recommender systems and ads.

### Why this makes sense:

* Users who click more frequently are more likely to convert.
* This is widely used in ads: **RFM (Recency–Frequency–Monetary) modeling**.

**q is NOT the conversion probability**, but a *soft signal* that higher-intent users tend to click or convert more.

---

# ## 2. What is `click_pos`?

### `click_pos` = position of the click inside a session

* `click_pos = 0` → the click occurred right at this item
* `click_pos = 7` → the click occurred 7 items later
* `click_pos = -1` → no click in the session

This is common in ad datasets like Criteo or Avazu, where you log:

* whether the user clicked here (click=1)
* how far the click was from this impression
* number of clicks in the session (`click_nb`)



$$
pCTR^T x + \beta q^T x - \lambda ||x||^2
$$

Maximise ctr, maximize cvr, and add regularization

In [12]:
q = 1 / (df['time_since_last_click'].replace(-1, 1e6) + 1)
q = q.to_numpy()


In [13]:
import cvxpy as cp
import numpy as np

cost = df['cost'].to_numpy()
pctr = np.array(pctr)
q = q.astype(float)

n = len(df)
x = cp.Variable(n, nonneg=True)

lambda_reg = 0.1
beta = 0.5
B = 200   # budget constraint

objective = cp.Maximize(
    pctr @ x + beta * (q @ x) - lambda_reg * cp.sum_squares(x)
)

constraints = [
    cost @ x <= B,
]

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.SCS)

print("Objective:", prob.value)
print("Chosen:", np.sum(x.value > 1e-6))


Objective: 124326.9118774325
Chosen: 50000


In [15]:
x.value[:10]

array([4.9791696 , 4.979173  , 4.97944864, 4.9791696 , 4.9791696 ,
       4.97917663, 4.97918102, 4.9791696 , 4.9791696 , 4.9791696 ])

In [16]:
print(prob.solver_stats)


SolverStats(solver_name='SCS', solve_time=0.19800331, setup_time=0.095557593, num_iters=50, extra_stats={'x': array([4.9791696 , 4.979173  , 4.97944864, ..., 4.97916882, 4.9791696 ,
       4.9791692 ]), 'y': array([0., 0., 0., ..., 0., 0., 0.]), 's': array([  4.9791696 ,   4.979173  ,   4.97944864, ...,   4.9791696 ,
         4.9791692 , 134.89762115]), 'info': {'status_val': 1, 'iter': 50, 'scale_updates': 0, 'scale': 0.1, 'pobj': -124326.91187743304, 'dobj': -124326.91199649239, 'res_pri': 3.686253340230024e-11, 'res_dual': 5.380116393141641e-08, 'gap': 0.00011905935703753314, 'res_infeas': nan, 'res_unbdd_a': 0.0008043310852176661, 'res_unbdd_p': 5.010312869132471e-06, 'comp_slack': 1.4053607143847265e-09, 'solve_time': 198.00331, 'setup_time': 95.557593, 'lin_sys_time': 129.548739, 'cone_time': 14.634548999999996, 'accel_time': 12.675417, 'rejected_accel_steps': 0, 'accepted_accel_steps': 0, 'status': 'solved'}})


Under the hood splitting cone solver was used.